# Notebook 04: Ingredient Enrichment BASED on product ingredient list

**Goal:** Check ingredient list coverage of our PAs cleaned ingredient dataset vs product dataset... also maybe expand our ingredient database to cover more products?

**Approach:**
1. Start with Paula's Choice database (2,530 ingredients)
2. Check product coverage we got with our ingredient dataset.
3. Add EU allergens (mandated labeling)
4. Add manual classifications (common ingredients)
5. Add pattern matching (plant extracts, oils, CI codes)
6. Compare before/after coverage

In [12]:
import pandas as pd
import numpy as np
from collections import Counter

## 1. Load Data

In [13]:
# Load products and Paula's Choice ingredient database
products = pd.read_parquet('../data/cleaned/skingen_products_lean_clean.parquet')
ingredients_db = pd.read_parquet('../data/cleaned/ingredients_cleaned.parquet')

print(f"Products: {len(products)}")
print(f"Paula's Choice ingredients: {len(ingredients_db)}")

Products: 10296
Paula's Choice ingredients: 2530


In [14]:
# Get all unique ingredients from products
all_product_ingredients = set()
for ing_list in products['ingredient_list']:
    all_product_ingredients.update(ing_list)

print(f"Unique ingredients in products: {len(all_product_ingredients)}")


Unique ingredients in products: 7825


In [15]:
# Quick look at data
print("Product columns:")
print(products.columns.tolist())

print("Ingredient columns:")
print(ingredients_db.columns.tolist())

Product columns:
['brand', 'name', 'type', 'country', 'ingredient_list', 'ingredient_count', 'claimed_concerns', 'positive_concerns', 'negative_concerns', 'condition_concerns', 'positive_count', 'negative_count', 'condition_count']
Ingredient columns:
['ingredient_name', 'rating', 'benefits', 'categories', 'info', 'functional_group']


## 2. Baseline Coverage (BEFORE Enrichment)

Check how many ingredients Paula's Choice covers.

In [16]:
# Count how often each ingredient appears in products
ingredient_frequency = Counter()

for ing_list in products['ingredient_list']:
    for ingredient in ing_list:
        ingredient_frequency[ingredient.lower()] += 1

print(f"Total ingredient occurrences counted: {sum(ingredient_frequency.values())}")

Total ingredient occurrences counted: 291811


In [17]:
# Check Paula's Choice coverage
paulas_names_lower = set(ingredients_db['ingredient_name'].str.lower())
matched_before = [ing for ing in all_product_ingredients if ing.lower() in paulas_names_lower]

print("BEFORE ENRICHMENT (Paula's Choice only):")
print(f"Matched: {len(matched_before)} / {len(all_product_ingredients)} ({len(matched_before)/len(all_product_ingredients)*100:.1f}%)")
print(f"Unknown: {len(all_product_ingredients) - len(matched_before)} ({(len(all_product_ingredients) - len(matched_before))/len(all_product_ingredients)*100:.1f}%)")

BEFORE ENRICHMENT (Paula's Choice only):
Matched: 1493 / 7825 (19.1%)
Unknown: 6332 (80.9%)


In [18]:
# Per-product coverage BEFORE
per_product_before = []
for ing_list in products['ingredient_list']:
    matched = sum(1 for ing in ing_list if ing.lower() in paulas_names_lower)
    coverage = (matched / len(ing_list)) * 100 if len(ing_list) > 0 else 0
    per_product_before.append(coverage)

print(f"Average per-product coverage: {np.mean(per_product_before):.1f}%")

Average per-product coverage: 77.3%


In [19]:
# Top 20 MISSING ingredients
print("Top 20 missing ingredients (by frequency):")

missing_sorted = []
for ingredient, count in ingredient_frequency.most_common():
    if ingredient not in ingredients_db['ingredient_name'].str.lower().values:
        missing_sorted.append((ingredient, count))

for ingredient, count in missing_sorted[:20]:
    print(f"{ingredient:45s}: {count:5d} products")

Top 20 missing ingredients (by frequency):
parfum                                       :  3241 products
2-hexanediol                                 :  2854 products
simmondsia chinensis seed oil                :   722 products
alcohol denat.                               :   696 products
glycyrrhiza glabra root extract              :   609 products
bha                                          :   529 products
rosmarinus officinalis leaf extract          :   529 products
lavandula angustifolia oil                   :   414 products
ci 77891                                     :   396 products
citral                                       :   367 products
curcuma longa root extract                   :   338 products
citrus aurantium bergamia fruit oil          :   332 products
melaleuca alternifolia leaf oil              :   331 products
pyrus malus fruit extract                    :   331 products
citrus aurantium dulcis peel oil             :   326 products
ci 77492                   

In [20]:
# Top 20 MATCHED ingredients
print("Top 20 matched ingredients (by frequency):")

matched_sorted = []
for ingredient, count in ingredient_frequency.most_common():
    if ingredient in ingredients_db['ingredient_name'].str.lower().values:
        matched_sorted.append((ingredient, count))

for ingredient, count in matched_sorted[:20]:
    print(f"{ingredient:45s}: {count:5d} products")

Top 20 matched ingredients (by frequency):
water                                        :  8961 products
glycerin                                     :  8003 products
phenoxyethanol                               :  4540 products
butylene glycol                              :  4303 products
ethylhexylglycerin                           :  4098 products
xanthan gum                                  :  3540 products
sodium hyaluronate                           :  3452 products
disodium edta                                :  3427 products
tocopherol                                   :  3242 products
citric acid                                  :  3086 products
caprylyl glycol                              :  2868 products
propanediol                                  :  2636 products
panthenol                                    :  2358 products
niacinamide                                  :  2295 products
sodium hydroxide                             :  2233 products
tocopheryl acetate         

## 3. Define Enrichment Sources

Add ingredients from EU regulations, manual classifications, and pattern matching.

In [21]:
# Functional group mapping (categories => functional groups) // reused from 03 notebook
FUNCTIONAL_GROUPS = {
    'Actives': ['Retinoids', 'Peptides', 'Exfoliant', 'Antioxidant', 'UV Filters',
                'Prebiotic/Probiotic/Postbiotic', 'Antibacterial'],
    'Support': ['Humectant', 'Emollient', 'Occlusive/Opacifying Agent', 'Plant Extracts'],
    'Utility': ['Emulsifier', 'Preservative', 'Solvent', 'pH Adjuster/Stabilizer',
                'Cleansing Agent', 'Chelating Agent'],
    'Sensory': ['Silicone', 'Polymer', 'Texture Enhancer', 'Fragrance: Synthetic and Natural',
                'Coloring Agent/Pigment', 'Absorbent', 'Film-Forming Agent',
                'Suspending/Dispersing Agent'],
    'Risks': ['Irritant']
}

def get_functional_groups(categories):
    """Convert categories to functional groups"""
    groups = []
    for category in categories:
        for group_name, group_cats in FUNCTIONAL_GROUPS.items():
            if category in group_cats and group_name not in groups:
                groups.append(group_name)
    return groups

# Test
test = get_functional_groups(['Antioxidant', 'Irritant'])
print(f"Test: {test}  (should be ['Actives', 'Risks'])")

Test: ['Actives', 'Risks']  (should be ['Actives', 'Risks'])


In [22]:
# EU Banned allergens (no longer permitted since 2021)
EU_BANNED = {
    'hydroxyisohexyl 3-cyclohexene carboxaldehyde',  # Lyral
    'atranol',
    'chloroatranol'
}

# EU 26 allergens (must be labeled separately)
EU_26 = {
    'alpha-isomethyl ionone', 'amyl cinnamal', 'amylcinnamyl alcohol',
    'anise alcohol', 'benzyl alcohol', 'benzyl benzoate', 'benzyl cinnamate',
    'benzyl salicylate', 'butylphenyl methylpropional', 'cinnamal',
    'cinnamyl alcohol', 'citral', 'citronellol', 'coumarin', 'eugenol',
    'farnesol', 'geraniol', 'hexyl cinnamal', 'hydroxycitronellal',
    'isoeugenol', 'limonene', 'linalool', 'methyl 2-octynoate',
    'evernia furfuracea extract', 'evernia prunastri extract'
}

# Manual known ingredients
KNOWN = {
    'parfum': {'rating': 'Worst', 'categories': ['Fragrance: Synthetic and Natural', 'Irritant']},
    'fragrance': {'rating': 'Worst', 'categories': ['Fragrance: Synthetic and Natural', 'Irritant']},
    'alcohol denat.': {'rating': 'Bad', 'categories': ['Solvent', 'Irritant']},
    '2-hexanediol': {'rating': 'Average', 'categories': ['Preservative', 'Solvent']},
    'bha': {'rating': 'Average', 'categories': ['Preservative', 'Antioxidant']},
}



print(f"EU Banned: {len(EU_BANNED)}")
print(f"EU 26: {len(EU_26)}")
print(f"Manual: {len(KNOWN)}")

EU Banned: 3
EU 26: 25
Manual: 5


## 4. Enrichment Function

One simple function that checks all sources in priority order.

In [23]:
def enrich_ingredient(ingredient_name):
    """
    Classify an ingredient using all available sources.
    Priority: Paula's => EU Banned => EU 26 => Manual => Pattern => Unknown
    """
    key = ingredient_name.lower()
    
    # 1. Paula's Choice
    match = ingredients_db[ingredients_db['ingredient_name'].str.lower() == key]
    if len(match) > 0:
        row = match.iloc[0]
        return {
            'ingredient_name': ingredient_name,
            'rating': row['rating'],
            'benefits': row['benefits'],
            'categories': row['categories'],
            'functional_group': row['functional_group'],
            'info': row['info'],
            'source': "Paula's Choice"
        }
    
    # 2. EU Banned
    if key in {x.lower() for x in EU_BANNED}:
        categories = ['Fragrance: Synthetic and Natural', 'Irritant']
        return {
            'ingredient_name': ingredient_name,
            'rating': 'Worst',
            'benefits': [],
            'categories': categories,
            'functional_group': get_functional_groups(categories),
            'info': ['EU Banned Allergen since 2021'],
            'source': 'EU Banned'
        }
    
    # 3. EU 26
    if key in {x.lower() for x in EU_26}:
        categories = ['Fragrance: Synthetic and Natural', 'Irritant']
        return {
            'ingredient_name': ingredient_name,
            'rating': 'Bad',
            'benefits': [],
            'categories': categories,
            'functional_group': get_functional_groups(categories),
            'info': ['EU list of 26 fragrance allergens requiring labeling'],
            'source': 'EU 26'
        }
    
    # 4. Manual known
    if key in KNOWN:
        data = KNOWN[key]
        return {
            'ingredient_name': ingredient_name,
            'rating': data['rating'],
            'benefits': [],
            'categories': data['categories'],
            'functional_group': get_functional_groups(data['categories']),
            'info': [],
            'source': 'Manual'
        }
    
    # 5. Pattern matching
    
    # Plant extracts (non-oil)
    if 'extract' in key and 'oil' not in key:
        categories = ['Plant Extracts','Antioxidant']
        return {
            'ingredient_name': ingredient_name,
            'rating': 'Good',
            'benefits': [],
            'categories': categories,
            'functional_group': get_functional_groups(categories),
            'info': [],
            'source': 'Pattern'
        }
    
    # Seed/nut oils (emollients)
    if 'seed oil' in key or 'kernel oil' in key or 'nut oil' in key:
        categories = ['Emollient', 'Plant Extracts']
        return {
            'ingredient_name': ingredient_name,
            'rating': 'Good',
            'benefits': ['Hydration'],
            'categories': categories,
            'functional_group': get_functional_groups(categories),
            'info': [],
            'source': 'Pattern'
        }
    
    # Essential oils (fragrant - irritants)
    fragrant_plants = ['lavandula', 'rosmarinus', 'eucalyptus', 'citrus', 
                       'melaleuca', 'mentha', 'bergamot', 'lemon', 'orange']
    if 'oil' in key and any(plant in key for plant in fragrant_plants):
        categories = ['Fragrance: Synthetic and Natural', 'Irritant', 'Plant Extracts']
        return {
            'ingredient_name': ingredient_name,
            'rating': 'Bad',
            'benefits': [],
            'categories': categories,
            'functional_group': get_functional_groups(categories),
            'info': [],
            'source': 'Pattern'
        }
    
    # CI color codes
    if key.startswith('ci '):
        categories = ['Coloring Agent/Pigment']
        return {
            'ingredient_name': ingredient_name,
            'rating': 'Average',
            'benefits': [],
            'categories': categories,
            'functional_group': get_functional_groups(categories),
            'info': [],
            'source': 'Pattern'
        }
    
    # Plant waters
    if 'water' in key and any(x in key for x in ['flower', 'leaf', 'fruit']):
        categories = ['Humectant', 'Plant Extracts','Antioxidant']
        return {
            'ingredient_name': ingredient_name,
            'rating': 'Good',
            'benefits': ['Hydration', 'Soothing'],
            'categories': categories,
            'functional_group': get_functional_groups(categories),
            'info': [],
            'source': 'Pattern'
        }
    
    # Butters
    if 'butter' in key:
        categories = ['Emollient']
        return {
            'ingredient_name': ingredient_name,
            'rating': 'Good',
            'benefits': ['Hydration'],
            'categories': categories,
            'functional_group': get_functional_groups(categories),
            'info': [],
            'source': 'Pattern'
        }
    
    # Unknown
    return None

In [24]:
# Test
test_cases = ['Niacinamide', 'citral', 'parfum', 'ci 77891', 'aloe vera extract']
for ing in test_cases:
    result = enrich_ingredient(ing)
    if result:
        print(f"{ing}: {result['source']} - {result['rating']} - {result['functional_group']} - {result['benefits']} - {result['categories']}")

Niacinamide: Paula's Choice - Best - ['Actives' 'Support'] - ['Anti-Aging' 'Pore Minimizer' 'Soothing'] - ['Antioxidant' 'Humectant']
citral: EU 26 - Bad - ['Sensory', 'Risks'] - [] - ['Fragrance: Synthetic and Natural', 'Irritant']
parfum: Manual - Worst - ['Sensory', 'Risks'] - [] - ['Fragrance: Synthetic and Natural', 'Irritant']
ci 77891: Pattern - Average - ['Sensory'] - [] - ['Coloring Agent/Pigment']
aloe vera extract: Pattern - Good - ['Support', 'Actives'] - [] - ['Plant Extracts', 'Antioxidant']


## 5. Apply Enrichment to All Ingredients
- ste p1 : Add ALL Paula's Choice ingredients (100% of them) to dataset
- step 2 : Add NEW ingredients from products (only if successfully enriched)

In [25]:
enriched_list = []

#Add ALL Paula's Choice ingredients (100% of them)
for idx, row in ingredients_db.iterrows():
    enriched_list.append({
        'ingredient_name': row['ingredient_name'],
        'rating': row['rating'],
        'benefits': row['benefits'],
        'categories': row['categories'],
        'functional_group': row['functional_group'],
        'info': row['info'],
        'source': "Paula's Choice"
    })

print(f"Added: {len(enriched_list)} ingredients")


Added: 2530 ingredients


In [26]:
# Enrich unknown ingredients from products
paulas_names_lower = set(ingredients_db['ingredient_name'].str.lower())
new_count = 0

for ing in all_product_ingredients:
    key = ing.lower()
    
    # Skip if already in Paula's Choice
    if key in paulas_names_lower:
        continue
    
    # Try to enrich with our mappings (EU/Manual/Pattern)
    data = enrich_ingredient(ing)
    
    # Only add if we successfully enriched it (not unknown)
    if data and data['source'] != "Paula's Choice":
        enriched_list.append(data)
        new_count += 1

print(f"Added: {new_count} new ingredients")

Added: 2455 new ingredients


In [27]:
# Create final dataframe
ingredients_enriched = pd.DataFrame(enriched_list)

print(f"Total enriched ingredients: {len(ingredients_enriched)}")
print(f"By source:")
print(ingredients_enriched['source'].value_counts())

Total enriched ingredients: 4985
By source:
source
Paula's Choice    2530
Pattern           2441
EU 26                9
Manual               4
EU Banned            1
Name: count, dtype: int64


## 6. Save Enriched Ingredient Database

In [28]:
# Save enriched database
ingredients_enriched.to_csv('../data/cleaned/ingredients_enriched.csv', index=False)
output_path = '../data/cleaned/ingredients_enriched.parquet'
ingredients_enriched.to_parquet(output_path, index=False)

print(f"Saved: {output_path}")
print(f"Shape: {ingredients_enriched.shape}")
print(f"Columns: {list(ingredients_enriched.columns)}")

Saved: ../data/cleaned/ingredients_enriched.parquet
Shape: (4985, 7)
Columns: ['ingredient_name', 'rating', 'benefits', 'categories', 'functional_group', 'info', 'source']


## 7. Coverage After Enrichment

In [29]:
# Load enriched ingredient database
ingredients_enriched = pd.read_parquet("../data/cleaned/ingredients_enriched.parquet")

print(f"Loaded ingredients: {ingredients_enriched.shape}")

Loaded ingredients: (4985, 7)


In [30]:
# Check coverage AFTER
enriched_names_lower = set(ingredients_enriched['ingredient_name'].str.lower())

# Count how many PRODUCT ingredients we can now match
matched_after = [ing for ing in all_product_ingredients if ing.lower() in enriched_names_lower]

print("AFTER ENRICHMENT:")
print(f"Matched: {len(matched_after)} / {len(all_product_ingredients)} ({len(matched_after)/len(all_product_ingredients)*100:.1f}%)")
print(f"Unknown: {len(all_product_ingredients) - len(matched_after)} ({(len(all_product_ingredients) - len(matched_after))/len(all_product_ingredients)*100:.1f}%)")


AFTER ENRICHMENT:
Matched: 3948 / 7825 (50.5%)
Unknown: 3877 (49.5%)


In [31]:
# Per-product coverage AFTER
per_product_after = []
for ing_list in products['ingredient_list']:
    matched = sum(1 for ing in ing_list if ing.lower() in enriched_names_lower)
    coverage = (matched / len(ing_list)) * 100 if len(ing_list) > 0 else 0
    per_product_after.append(coverage)

print(f"Average per-product coverage: {np.mean(per_product_after):.1f}%")

Average per-product coverage: 90.3%


## 8. Before vs After Comparison

In [42]:
print("IMPROVEMENT SUMMARY")

print("Unique ingredient coverage:")
# VOOR: 
print(f"Before: {len(matched_before)} ({len(matched_before)/len(all_product_ingredients)*100:.1f}%)")

# NA: 
print(f"After: {len(matched_after)} ({len(matched_after)/len(all_product_ingredients)*100:.1f}%)")

# GAIN:
gain_count = len(matched_after) - len(matched_before)
gain_pct = (len(matched_after) - len(matched_before)) / len(all_product_ingredients) * 100
print(f"Gain: +{gain_count} ({gain_pct:+.1f}%)")


IMPROVEMENT SUMMARY
Unique ingredient coverage:
Before: 1493 (19.1%)
After: 3948 (50.5%)
Gain: +2455 (+31.4%)


In [36]:
print("Per-product coverage (Average):")
print(f"Before:{np.mean(per_product_before):.1f}%")
print(f"After:{np.mean(per_product_after):.1f}%")
print(f"Gain:{np.mean(per_product_after) - np.mean(per_product_before):+.1f}%")

Per-product coverage (Average):
Before:77.3%
After:90.3%
Gain:+13.0%


## Summary

Successfully enriched the ingredient database by:
- Adding EU regulatory data (allergens)
- Adding manual classifications (common ingredients)
- Adding pattern matching (plant-based ingredients)

IMPROVEMENT SUMMARY

Unique ingredient coverage:
- Before: 1493 (19.1%)
- After: 3948 (50.5%)
- Gain: +2455 (+31.4%)

Per-product coverage (Average):
- Before:77.3%
- After:90.3%
- Gain:+13.0%

Making the dataset suitable for building product recommendations based on ingredient intelligence in the future.